In [ ]:
import sys, os
# Ноутбук лежит в notebooks/, а utils.py — в src/. Добавляем путь.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc
import arviz as az
import pytensor.tensor as pt

from utils import (
    create_bipartite_bayesian_network_cond,
    create_bipartite_bayesian_network_nocond,
    f_2, f_3, f_4, f_5,
    ratio, RATIO_KINDS,
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Быстрая проверка: сумма f_2+f_3+f_4+f_5 равна 1 на сетке
x_grid = np.linspace(0.05, 0.95, 5)
totals = f_2(x_grid) + f_3(x_grid) + f_4(x_grid) + f_5(x_grid)
print('f_2+f_3+f_4+f_5 на сетке:', totals.round(6))

print('\nratio в угловых точках:')
for name in RATIO_KINDS:
    print(f'  {name:11s}:', {
        '(s=0, d=1)':  ratio(0.0, 1.0, kind=name),
        '(s=.5,d=.5)': ratio(0.5, 0.5, kind=name),
        '(s=1, d=0)':  ratio(1.0, 0.0, kind=name),
        '(s=0, d=0)':  ratio(0.0, 0.0, kind=name),
        '(s=1, d=1)':  ratio(1.0, 1.0, kind=name),
    })

In [ ]:
def generate_data(students_size, items_size, random_state=42):
    np.random.seed(random_state)
    return np.random.randint(low=2, high=5,
                             size=students_size * items_size).reshape(students_size, items_size)

In [ ]:
data = generate_data(4, 3)
data

In [ ]:
trace, model = create_bipartite_bayesian_network_cond(
    ratings_matrix=data,
    student_alpha=2, student_beta=2,
    item_alpha=2,   item_beta=4,
    ratio_kind='current',  # попробуй 'legacy_sd' или 'sigmoid' для сравнения
    draws=1000, tune=2000, chains=4, cores=2, target_accept=0.95,
)

In [ ]:
def show_trace(trace):
    fig, axes = plt.subplots(ncols=2, nrows=7, figsize=(14, 20))
    az.plot_trace(trace, compact=False, legend=True, axes=axes)
    axes = axes.flatten()
    for i in range(0, len(axes), 2):
        axes[i].set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

In [ ]:
show_trace(trace)